In [5]:
import asyncio
from threading import Thread
from playwright.async_api import async_playwright

result = {}

async def test_browser():
    pw = await async_playwright().start()
    browser = await pw.chromium.launch(channel="msedge", headless=False)

    page = await browser.new_page()
    await page.goto("https://www.google.com")

    result["title"] = await page.title()

    await browser.close()
    await pw.stop()

def runner():
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    asyncio.run(test_browser())

thread = Thread(target=runner)
thread.start()
thread.join()

print(result)

{'title': 'Google'}


In [6]:
import asyncio
from threading import Thread
from queue import Queue
from playwright.async_api import async_playwright

class PlaywrightWorker:
    def __init__(self):
        self.loop = None
        self.thread = Thread(target=self._run, daemon=True)
        self.ready = Queue()
        self.thread.start()
        self.ready.get()

    def _run(self):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        self.loop = asyncio.new_event_loop()
        asyncio.set_event_loop(self.loop)
        self.ready.put(True)
        self.loop.run_forever()

    def run(self, coro):
        future = asyncio.run_coroutine_threadsafe(coro, self.loop)
        return future.result()

worker = PlaywrightWorker()

In [7]:
async def start_browser():
    global pw, context, page

    pw = await async_playwright().start()

    context = await pw.chromium.launch_persistent_context(
        user_data_dir="./browser_profile",
        channel="msedge",
        headless=False
    )

    page = context.pages[0] if context.pages else await context.new_page()

worker.run(start_browser())

In [ ]:
WEB_URL = "https://rate.bot.com.tw/xrt?Lang=zh-TW"

worker.run(
    page.goto(
        WEB_URL,
        wait_until="domcontentloaded",
        timeout=60000
    )
)

In [ ]:
worker.run(page.title())

In [6]:
from pathlib import Path
from playwright.sync_api import sync_playwright
import pandas as pd

WEB_URL = "https://rate.bot.com.tw/xrt?Lang=zh-TW"
PROFILE_DIR = Path("./browser_profile")
CSV_FILE = Path("rate.csv")

with sync_playwright() as p:
    context = p.chromium.launch_persistent_context(
        user_data_dir=str(PROFILE_DIR),
        headless=False
    )

    page = context.pages[0] if context.pages else context.new_page()

    print("Opening:", WEB_URL)
    page.goto(WEB_URL, wait_until="domcontentloaded", timeout=60000)

    for _ in range(60):
        title = page.title()
        print("Title:", title)

        if "Challenge Validation" not in title:
            break

        page.wait_for_timeout(1000)
    else:
        context.close()
        raise RuntimeError("Challenge 沒有完成")

    print("Challenge passed")
    print("Current URL:", page.url)

    page.wait_for_selector('table[title="牌告匯率"]', timeout=30000)
    print("Exchange rate page loaded")

    csv_link = page.get_by_role("link", name="下載 Excel (CSV) 檔")

    with page.expect_download(timeout=30000) as download_info:
        csv_link.click()

    download = download_info.value
    download.save_as(CSV_FILE)

    print("CSV saved:", CSV_FILE)

    context.close()

data = CSV_FILE.read_bytes()

if b"Challenge Validation" in data:
    raise RuntimeError("下載內容是 Challenge HTML，不是 CSV")

df = pd.read_csv(CSV_FILE, encoding="utf-8-sig")
print(df.head())

Error: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.